# Empirical Exploratory Data Analysis (EDA)
## Short-Term Electricity Demand Forecasting on Sri Lanka National Grid
**Author**: P. R. T. Sandaruwan (S25021963)

This notebook performs exploratory data analysis on 15-minute resolution utility telemetry data, syncing directly with the underlying `src/` modules (`src.data_ingestion` and `src.preprocess`).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_ingestion import load_raw_telemetry, validate_time_series_integrity
from src.preprocess import preprocess_data, create_lag_features, encode_cyclical_features

sns.set_theme(style="whitegrid")

### 1. Data Ingestion & Time-Series Integrity Verification
Loading raw CEB telemetry and executing automated time-series grid validation.

In [ ]:
raw_csv_path = os.path.join("..", "data", "raw", "load_forecasting_dataset_corrected.csv")
if not os.path.exists(raw_csv_path):
    raw_csv_path = os.path.join("data", "raw", "load_forecasting_dataset_corrected.csv")

telemetry_df = load_raw_telemetry(raw_csv_path)
validate_time_series_integrity(telemetry_df)

### 2. Loading Preprocessed Feature Matrix
Inspecting preprocessed telemetry containing autoregressive load lags (t-1, t-2, t-24, t-168) and cyclical temporal encodings.

In [ ]:
processed_csv_path = os.path.join("..", "data", "processed", "load_forecasting_dataset_processed.csv")
if not os.path.exists(processed_csv_path):
    processed_csv_path = os.path.join("data", "processed", "load_forecasting_dataset_processed.csv")

df = pd.read_csv(processed_csv_path)
print(f"Preprocessed Dataset Shape: {df.shape}")
df.head()

### 3. Autoregressive Lag Autocorrelation Matrix
Analyzing autocorrelation between historical load lags and target demand.

In [ ]:
lag_cols = [c for c in df.columns if "lag_" in c] + ["Load Demand (kW)"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[lag_cols].corr(), annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Autoregressive Lag Autocorrelation Heatmap Matrix", fontweight="bold")
plt.show()

### 4. Ambient Temperature vs. Load Demand Distribution
Visualizing cooling demand scaling against regional weather attributes.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df.sample(5000, random_state=42), x="Temperature (°C)", y="Load Demand (kW)", alpha=0.4, color="#D95319")
plt.title("Cooling Demand Scaling vs. Ambient Temperature (°C)", fontweight="bold")
plt.xlabel("Ambient Temperature (°C)")
plt.ylabel("Load Demand (kW)")
plt.show()